# 01 Data Ingestion
---

## Información
### Objetivo
Este notebook implementa la primera etapa del pipeline RAG: la adquisición y preparación de la base documental. Su propósito es obtener automáticamente los documentos oficiales de la Municipalidad de Valdivia, extraer su contenido y realizar un proceso de limpieza y normalización que permita su posterior procesamiento.
### Objetivos específicos
* Identificar las fuentes documentales del sistema.
* Realizar la extracción automática de documentos mediante scraping.
* Descargar archivos en distintos formatos (PDF, HTML y DOCX).
* Extraer el contenido textual preservando la mayor cantidad de información posible.
* Eliminar elementos irrelevantes (encabezados, pies de página, espacios redundantes, caracteres especiales, etc.).
* Normalizar el texto para facilitar las etapas posteriores del pipeline.
* Almacenar los documentos limpios para su utilización en el proceso de chunking.
### Entradas
* Sitio web DOM Digital de la Municipalidad de Valdivia.
* Portal de Transparencia Municipal.
* Documentos oficiales en formato PDF, HTML y DOCX.
### Salidas
* Documentos descargados en data/raw/.
* Texto limpio y normalizado almacenado en data/processed/.
* Registro de la extracción y del procesamiento realizado.

---
### Parte 1. Document Ingestion

In [27]:
import sys
from pathlib import Path


# Add project root to path
root = Path.cwd().resolve()
if not (root / "src").exists():
    for parent in root.parents:
        if (parent / "src").exists():
            root = parent
            break
sys.path.insert(0, str(root))

from src.ingestion.scraper import download_pdf

output_dir = Path("../data/raw/muni_valdivia/planes/")


url = (
    "https://munivaldivia.cl/wp-content/uploads/2025/11/"
    "Plan-Comunal-de-Emergencia-2025-2.pdf"
)

pdf_path = download_pdf(
    url=url,
    output_dir=output_dir
)


pdf_path

WindowsPath('../data/raw/muni_valdivia/planes/Plan-Comunal-de-Emergencia-2025-2.pdf')

In [28]:
pdf_path.exists()

True

In [29]:
pdf_path.stat().st_size

1256600

In [30]:
# Install the required library for PDF processing
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import src.ingestion.loaders

document = src.ingestion.loaders.load_pdf(pdf_path)

print(document.id)
print(document.title)
print(document.file_type)
print(document.metadata)
print(document.content[:500])


8d184dd3-4226-4cd5-9310-3dd74cd9940f
Plan-Comunal-de-Emergencia-2025-2
pdf
{'pages': 33}
 
Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página  
1 de 33 
Fecha: Agosto-2025 
 
    
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
PLAN COMUNAL DE EMERGENCIA 
VALDIVIA 
 
 
 
ANEXO – PLAN POR AMENAZA INCENDIO FORESTAL 
ANEXO - PLAN POR AMENAZA REMOCIÓN EN MASA 
ANEXO- PLAN POR AMENAZA TSUNAMI 
ANEXO – PLAN POR AMENAZA DE INUNDACIÓN POR DESBORDE 
DEL RÍO CALLE CALLE 
 
 
 
 
 
 
 
 
 
 
 
 
Aprobado mediante Decreto Alcaldicio N° 10952


In [32]:
document.title

'Plan-Comunal-de-Emergencia-2025-2'

In [33]:
document.file_type

'pdf'

In [34]:
document.metadata

{'pages': 33}

---
## Parte 2. Document Cleaning

In [35]:
from src.processing.cleaner import clean_document

In [36]:
cleaned_document = clean_document(document)

In [37]:
print(cleaned_document.content[:1000])

Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página 
1 de 33 
Fecha: Agosto-2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
PLAN COMUNAL DE EMERGENCIA 
VALDIVIA 
 
 
 
ANEXO – PLAN POR AMENAZA INCENDIO FORESTAL 
ANEXO - PLAN POR AMENAZA REMOCIÓN EN MASA 
ANEXO- PLAN POR AMENAZA TSUNAMI 
ANEXO – PLAN POR AMENAZA DE INUNDACIÓN POR DESBORDE 
DEL RÍO CALLE CALLE 
 
 
 
 
 
 
 
 
 
 
 
 
Aprobado mediante Decreto Alcaldicio N° 10952/2025 
 
 
 
 
 
 
Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página 
2 de 33 
Fecha: Agosto-2025 
 
Índice 
 
N° Contenido Página 
1 Introducción 3 
1.1 Objetivos General y Específicos 4 
1.2 Cobertura, Amplitud y Alcance 4 
1.3 Relación con Otros Planes 5 
2 Descripción del Riesgo Comunal (Amenazas, Vulnerabilidades y Exposición) 5 
3 Activación del Plan Comunal de Emergencia 10 
4 Roles y Funciones 10 
5 Comité Comunal para la G

In [38]:
print(cleaned_document.metadata)

{'pages': 33}


In [39]:
print(cleaned_document.title)
print(cleaned_document.source)
print(cleaned_document.file_type)

Plan-Comunal-de-Emergencia-2025-2
..\data\raw\muni_valdivia\planes\Plan-Comunal-de-Emergencia-2025-2.pdf
pdf


In [40]:
print("===== ORIGINAL =====")
print(document.content[:500])

print("\n===== CLEANED =====")
print(cleaned_document.content[:500])

===== ORIGINAL =====
 
Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página  
1 de 33 
Fecha: Agosto-2025 
 
    
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
PLAN COMUNAL DE EMERGENCIA 
VALDIVIA 
 
 
 
ANEXO – PLAN POR AMENAZA INCENDIO FORESTAL 
ANEXO - PLAN POR AMENAZA REMOCIÓN EN MASA 
ANEXO- PLAN POR AMENAZA TSUNAMI 
ANEXO – PLAN POR AMENAZA DE INUNDACIÓN POR DESBORDE 
DEL RÍO CALLE CALLE 
 
 
 
 
 
 
 
 
 
 
 
 
Aprobado mediante Decreto Alcaldicio N° 10952

===== CLEANED =====
Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página 
1 de 33 
Fecha: Agosto-2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
PLAN COMUNAL DE EMERGENCIA 
VALDIVIA 
 
 
 
ANEXO – PLAN POR AMENAZA INCENDIO FORESTAL 
ANEXO - PLAN POR AMENAZA REMOCIÓN EN MASA 
ANEXO- PLAN POR AMENAZA TSUNAMI 
ANEXO – PLAN POR AMENAZA DE INUNDACIÓN POR DESBORDE 
DEL RÍO CALLE CALLE 
 
 
 
 
 
 
 
 
 
 
 
 
Aprobado

In [41]:
len(document.content)

90760

In [42]:
repr(document.content[:200])

"' \\nIlustre Municipalidad de Valdivia PLANTILLA \\nVERSION: 01 \\nPLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA \\nPágina  \\n1 de 33 \\nFecha: Agosto-2025 \\n \\n    \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\nPLAN CO'"

In [43]:
len(cleaned_document.content)

90189

In [44]:
repr(cleaned_document.content[:200])

"'Ilustre Municipalidad de Valdivia PLANTILLA \\nVERSION: 01 \\nPLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA \\nPágina \\n1 de 33 \\nFecha: Agosto-2025 \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\n \\nPLAN COMUNAL '"

---
## ✅ Conclusiones

En esta sección aprendimos que:

- Un documento puede descargarse desde una URL oficial.
- El loader transforma el PDF en un objeto Document.
- El cleaner prepara el texto para las siguientes etapas.
- Cada módulo tiene una única responsabilidad dentro del pipeline.